# MIMIC-IV revision analyses

Runtime → Run all (about 2–3 minutes). Downloads `mimic_revision_results.xlsx` (aggregate only).

In [ ]:
# ---- Settings ---------------------------------------------------------------
PROJECT  = "ctheg-scores"
SEED     = 20261112
B        = 500     # FINAL run: bootstrap reps for primary estimates
B_SENS   = 200     # FINAL run: reps for sensitivity / 65+ / Oaxaca / by-destination
OUTFILE  = "aim1_results.xlsx"
MIN_CELL = 11      # aggregate counts below this are suppressed in exported tables

import time, warnings
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # convergence is checked explicitly below

class NotConverged(Exception):
    pass

def check(res, label):
    ok = res.mle_retvals.get("converged", True) if hasattr(res, "mle_retvals") and res.mle_retvals else getattr(res, "converged", True)
    if not ok:
        raise NotConverged(label)
    return res
pd.set_option("display.width", 200, "display.max_columns", 50)
print("statsmodels", statsmodels.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# ---- Load the analysis views from BigQuery (no file downloads) ---------------
from google.colab import auth
auth.authenticate_user()          # sign in with the Google account linked to PhysioNet
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

COLS = """ins_cat, dc_group, post_icu_days, any_post_icu_stay, dc_ama, dc_acute_transfer,
          time_valid_strict, age_at_admit, gender, race_cat, sofa_day1, cci_no_age,
          adm_type_cat, invasive_vent, vasopressor, icu_los_days_total, first_careunit,
          anchor_year_group, language_cat"""

raw18 = client.query(f"SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_primary_18_64`").to_dataframe()
raw65 = client.query(f"SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_older_65plus`").to_dataframe()
print("18-64:", raw18.shape, "| 65+:", raw65.shape)

In [ ]:
# ---- Variable preparation -----------------------------------------------------
DC_LEVELS = ["home", "home_health", "snf", "rehab", "ltach", "other"]   # code 0 = home (reference)

ICU_MAP = {
    "Cardiac Vascular Intensive Care Unit (CVICU)":       "Cardiac",
    "Coronary Care Unit (CCU)":                           "Cardiac",
    "Medical Intensive Care Unit (MICU)":                 "Medical",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)":   "Mixed",
    "Surgical Intensive Care Unit (SICU)":                "Surgical/Trauma",
    "Trauma SICU (TSICU)":                                "Surgical/Trauma",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)":    "Neuro",
    "Neuro Intermediate":                                 "Neuro",
    "Neuro Stepdown":                                     "Neuro",
}   # anything else (PACU, Medicine, 'Intensive Care Unit (ICU)', small surgical units; ~60 patients) -> 'Mixed'
#     (merged 2026-11-12: a separate tiny group caused non-convergence in bootstrap resamples)

REF = {"race_cat": "White", "adm_type_cat": "Emergency", "icu_type": "Medical",
       "anchor_year_group": "2008 - 2010", "language_cat": "English"}

def prep(raw, exposure_levels):
    d = raw[raw["ins_cat"].isin(exposure_levels)].copy()
    d["icu_type"]    = d["first_careunit"].map(ICU_MAP).fillna("Mixed")
    d["female"]      = (d["gender"] == "F").astype(int)
    d["age_c"]       = (d["age_at_admit"] - 50) / 10            # per 10 years, centred at 50
    d["log_icu_los"] = np.log1p(d["icu_los_days_total"].astype(float))
    d["vent"]        = d["invasive_vent"].astype(int)
    d["pressor"]     = d["vasopressor"].astype(int)
    d["any_stay"]    = d["any_post_icu_stay"].astype(int)
    d["days_total"]  = np.where(d["any_stay"] == 1, d["post_icu_days"].astype(float), 0.0)  # two-part: <=4 h counted as 0
    d["post_icu_days"] = d["post_icu_days"].astype(float)
    d["sofa_day1"]   = d["sofa_day1"].astype(float)
    d["cci_no_age"]  = d["cci_no_age"].astype(float)
    d["facility"]    = d["dc_group"].isin(["snf", "rehab", "ltach"]).astype(int)
    for c in ["dc_ama", "dc_acute_transfer", "time_valid_strict"]:
        d[c] = d[c].astype(bool)
    d["dc_code"] = pd.Categorical(d["dc_group"], categories=DC_LEVELS).codes
    d["ins"] = pd.Categorical(d["ins_cat"], categories=exposure_levels)   # first level = reference
    for c, ref in REF.items():                                            # fixed level order, reference first
        levels = [ref] + sorted(x for x in d[c].dropna().unique() if x != ref)
        d[c] = pd.Categorical(d[c], categories=levels)
    n_missing = d[["sofa_day1", "cci_no_age", "dc_code"]].isna().sum()
    assert (d["dc_code"] >= 0).all(), "unexpected discharge group"
    print(f"n = {len(d):,} | missing:", n_missing.to_dict())
    return d.dropna(subset=["sofa_day1", "cci_no_age"]).reset_index(drop=True)

COV = ("age_c + I(age_c**2) + female + C(race_cat) + sofa_day1 + cci_no_age + C(adm_type_cat)"
       " + vent + pressor + log_icu_los + C(icu_type) + C(anchor_year_group)")
COV_LANG = COV + " + C(language_cat)"
EXPO = "C(ins)"

d18 = prep(raw18, ["Private", "Medicaid"])
d18["medicaid"] = (d18["ins_cat"] == "Medicaid").astype(int)
print(d18["ins"].value_counts())

In [ ]:
import warnings, numpy as np, pandas as pd, statsmodels.formula.api as smf
warnings.filterwarnings("ignore")
rawR = client.query(f"""SELECT v.*, CAST(s.housing_econ_icd AS INT64) AS housing, CAST(s.sud_icd AS INT64) AS sud,
                               CAST(s.smi_icd AS INT64) AS smi
                        FROM `{PROJECT}.ins_icu.aim1_primary_18_64` v
                        LEFT JOIN `{PROJECT}.ins_icu.aim1_s4_flags` s USING (hadm_id)""").to_dataframe()
dR = prep(rawR, ["Private", "Medicaid"])
dR["medicaid"] = (dR["ins_cat"] == "Medicaid").astype(int)
adm = pd.to_datetime(dR["admittime"]).dt.normalize(); dis = pd.to_datetime(dR["dischtime"]).dt.normalize()
dR["los_tx"] = ((dis - adm).dt.days).clip(lower=1).astype(float)
for k in ["home", "home_health", "snf", "rehab", "ltach", "other"]:
    dR[f"y_{k}"] = (dR["dc_group"] == k).astype(float) * 100
dR["y_facility"] = dR["dc_group"].isin(["snf", "rehab", "ltach"]).astype(float) * 100
for c in ["housing", "sud", "smi"]:
    dR[c] = dR[c].fillna(0).astype(int)
print(f"n = {len(dR):,}; Medicaid {dR['medicaid'].sum():,}")

M0 = "age_c + I(age_c**2) + female + C(race_cat) + C(adm_type_cat) + C(anchor_year_group)"
M1 = M0 + " + cci_no_age"
M2 = M1 + " + sofa_day1 + vent + pressor"
M3 = M2 + " + log_icu_los + C(icu_type)"            # = paper covariate set
M4 = M3 + " + housing + sud + smi"
OUTS = ["y_home", "y_home_health", "y_snf", "y_rehab", "y_ltach", "y_other", "y_facility", "los_tx", "days_total"]
rows = []
for name, cov in [("M0 admission-time characteristics", M0), ("M1 + Charlson index", M1),
                  ("M2 + first-day SOFA, ventilation, vasopressors", M2), ("M3 + ICU length of stay and type (paper set)", M3),
                  ("M4 + diagnosis-coded social-needs proxies", M4)]:
    for y in OUTS:
        r = smf.ols(f"{y} ~ medicaid + {cov}", data=dR).fit(cov_type="HC1")
        b, se = r.params["medicaid"], r.bse["medicaid"]
        rows.append(dict(model=name, outcome=y, n=int(r.nobs), diff=b, lo=b - 1.96 * se, hi=b + 1.96 * se, se=se))
ladder = pd.DataFrame(rows)
print(ladder.pivot_table(index="model", columns="outcome", values="diff", sort=False).round(2).to_string())

In [ ]:
# ---- IPW diagnostics (paper covariates; stabilized weights truncated at 1st/99th percentiles)
import patsy
ps = smf.logit(f"medicaid ~ {COV}", dR).fit(disp=0, maxiter=100).predict(dR).values
t = dR["medicaid"].values
pt = t.mean()
w = np.where(t == 1, pt / ps, (1 - pt) / (1 - ps))
lo, hi = np.quantile(w, [0.01, 0.99]); w = np.clip(w, lo, hi)
ess = lambda x: x.sum() ** 2 / (x ** 2).sum()
X = patsy.dmatrix(COV, dR, return_type="dataframe").drop(columns="Intercept")
def smd(wt):
    m1 = np.average(X[t == 1], axis=0, weights=wt[t == 1]); m0 = np.average(X[t == 0], axis=0, weights=wt[t == 0])
    s = np.sqrt((X[t == 1].var().values + X[t == 0].var().values) / 2)
    return np.abs((m1 - m0) / np.where(s > 0, s, np.nan))
diag = pd.DataFrame([
    dict(item="PS Medicaid p5/p50/p95", value=" / ".join(f"{np.quantile(ps[t == 1], q):.3f}" for q in [.05, .5, .95])),
    dict(item="PS private p5/p50/p95", value=" / ".join(f"{np.quantile(ps[t == 0], q):.3f}" for q in [.05, .5, .95])),
    dict(item="Share of Medicaid with PS outside private range, %", value=f"{100 * ((ps[t == 1] < ps[t == 0].min()) | (ps[t == 1] > ps[t == 0].max())).mean():.2f}"),
    dict(item="Weight min / max after truncation", value=f"{w.min():.2f} / {w.max():.2f}"),
    dict(item="Effective sample size, Medicaid (n)", value=f"{ess(w[t == 1]):,.0f} ({(t == 1).sum():,})"),
    dict(item="Effective sample size, private (n)", value=f"{ess(w[t == 0]):,.0f} ({(t == 0).sum():,})"),
    dict(item="Max absolute SMD before / after weighting", value=f"{np.nanmax(smd(np.ones_like(w))):.3f} / {np.nanmax(smd(w)):.3f}"),
])
print(diag.to_string(index=False))
with pd.ExcelWriter("mimic_revision_results.xlsx") as xw:
    ladder.to_excel(xw, sheet_name="ladder", index=False)
    diag.to_excel(xw, sheet_name="ipw_diagnostics", index=False)
from google.colab import files
files.download("mimic_revision_results.xlsx")